# VisiumHD reconstruction impact: Fibroblast, Mono/Macro and T

Read the first half for **what changed**, and the second half for **where those
changes occur**. Figures are followed by summaries; analysis tables and figures
are saved as each block completes. The source files and exact cohorts are audited
below. Each route keeps its own spatial and expression semantics.

1. [Comparison foundation](#1.-Comparison-foundation-—-what-are-we-comparing?)
2. [Overall changes](#2.-Overall-changes-—-what-changed?)
3. Spatial localization: anatomy, changed units, expression and diversity
4. Regions: state, extent, reliability and scale sensitivity
5. Cross-parent summary and evidence boundary

Design reference: `docs/design/reconstruction-impact/README.md`.

## 1. Comparison foundation — what are we comparing?

**Method:** [Comparison foundation](../../../docs/design/reconstruction-impact/input-views.md#comparison-foundation)

### 1.1 Route semantics and carrier audit

In [ ]:
import os
import sys
import json
from types import SimpleNamespace
from pathlib import Path

os.environ["TQDM_DISABLE"] = "1"  # Disable progress at the producer; retain warnings.
os.environ.setdefault("KMP_WARNINGS", "0")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("KMP_USE_SHM", "0")
os.environ.setdefault("KMP_AFFINITY", "disabled")

PROJECT_ROOT = Path(os.environ.get("REVISE_REPOSITORY_ROOT", Path.cwd())).resolve()
if not (PROJECT_ROOT / "configs").is_dir():
    PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from revise.analysis.reconstruction_impact import load_reconstruction_impact_config

ROOT = PROJECT_ROOT

from reproduce.case.reconstruction_impact.notebook_helpers import (
    NotebookFigures, coordinates, compact_iqr, extent_table, raw_level2_mapping,
    plot_threshold_reliability, prepare_aucell_overall, prepare_aucell_parent,
    prepare_route_inputs, prepare_expression_views, save_partition_evidence,
    save_anatomy_evidence, save_spatial_feature_values, prepare_local_state,
    save_local_state_evidence, save_final_evidence,
)

In [ ]:
CONFIG = load_reconstruction_impact_config(ROOT / "configs/analysis/reconstruction_impact_visiumhd_p1crc.yaml")
OUTPUT_DIR = Path(os.environ.get("REVISE_ANALYSIS_OUTPUT_ROOT", ROOT / CONFIG["output"]["dir"]))
SAMPLE_ID = CONFIG["sample"]["id"]
LEVEL1 = CONFIG["context"]["level1_column"]
SEED = int(CONFIG["partition_change"]["random_state"])
VISIUMHD_SAMPLE_N_UNITS = 30_000
USE_FULL_VISIUMHD_COHORT = False
CELL_EQUIVALENT_UM = 8.0
PARENT_SOURCE = {"Fibroblast": "Fibroblast", "Mono_Macro": "Mono/Macro", "T": "T"}
SOURCE_FILES = {"raw": ROOT / CONFIG["context"]["h5ad"], "raw_level2_reference": ROOT / CONFIG["raw_level2_mapping"]["reference_h5ad"]}
raw_context = ad.read_h5ad(SOURCE_FILES["raw"], backed="r")
full_coordinates = coordinates(raw_context)
full_level1 = raw_context.obs[LEVEL1].astype(str).copy()
if not raw_context.obs_names.is_unique or not raw_context.var_names.is_unique:
    raise ValueError("Raw observation and gene axes must be unique")
if not np.isfinite(full_coordinates.to_numpy()).all():
    raise ValueError("Raw spatial coordinates must be finite")
TISSUE_COORDINATES_UM = full_coordinates * CONFIG["spatial_region"]["microns_per_coordinate"]
TISSUE_XLIM = tuple(TISSUE_COORDINATES_UM["x"].agg(["min", "max"]))
TISSUE_YLIM = tuple(TISSUE_COORDINATES_UM["y"].agg(["min", "max"]))
ORIGIN_UM = (TISSUE_XLIM[0], TISSUE_YLIM[0])
figures = NotebookFigures(OUTPUT_DIR, TISSUE_XLIM, TISSUE_YLIM, ORIGIN_UM, CELL_EQUIVALENT_UM)
RUNS = {}
input_rows = [{"role": "Raw full context", "n_units": raw_context.n_obs, "n_genes": raw_context.n_vars, "spatial_axis": True}]
# global_ids and parent IDs are sampled inside prepare_route_inputs.

<!-- impact-node:1.1 -->
**Carrier and route semantics — batch interpretation**

Method: [input-views.md#comparison-foundation](../../../docs/design/reconstruction-impact/input-views.md#comparison-foundation)

### 1.2 Cohort selection and coordinate audit

**Method:** [Comparison foundation](../../../docs/design/reconstruction-impact/input-views.md#comparison-foundation)

In [ ]:
def deterministic_same_id_sample(ids, limit):
    ids = pd.Index(ids); return ids if USE_FULL_VISIUMHD_COHORT or len(ids) <= limit else pd.Index(np.random.default_rng(SEED).choice(ids.to_numpy(), size=limit, replace=False))
prepared = prepare_route_inputs(
    CONFIG, ROOT, raw_context, full_coordinates, full_level1, figures,
    SOURCE_FILES, input_rows, seed=SEED,
    sample_limit=VISIUMHD_SAMPLE_N_UNITS,
    use_full_visiumhd_cohort=USE_FULL_VISIUMHD_COHORT,
    parent_source=PARENT_SOURCE, sample_fn=deterministic_same_id_sample,
)
SOURCE_FILES = prepared["source_files"]; RUNS = prepared["runs"]
GLOBAL_INPUTS = prepared["global_inputs"]; input_rows = prepared["input_rows"]
display(pd.DataFrame(input_rows))

<!-- impact-node:1.2 -->
**Comparison scope and cell types — batch interpretation**

Method: [input-views.md#cohort-presentation](../../../docs/design/reconstruction-impact/input-views.md#cohort-presentation)

### 1.3 Pairing and spatial coordinates

**Method:** [Comparison foundation](../../../docs/design/reconstruction-impact/input-views.md#comparison-foundation)

<!-- impact-node:1.3 -->
**Pairing and spatial coordinates — batch interpretation**

Method: [input-views.md#pairing-presentation](../../../docs/design/reconstruction-impact/input-views.md#pairing-presentation)

### 1.4 Expression views and gene availability

**Method:** [Comparison foundation](../../../docs/design/reconstruction-impact/input-views.md#comparison-foundation)

In [ ]:
# Xenium expression_h5ad is projected inside this complete-gene helper.
GENE_AVAILABILITY, input_rows, EXPRESSION_AUDITS = prepare_expression_views(
    RUNS, CONFIG, ROOT, figures, SOURCE_FILES, input_rows,
)
for table in EXPRESSION_AUDITS:
    display(table)
display(GENE_AVAILABILITY.groupby(
    ["scope", "side", "status"], sort=False
).size().rename("genes").reset_index())

<!-- impact-node:1.4 -->
**Expression views and full gene space — batch interpretation**

Method: [input-views.md#gene-space-presentation](../../../docs/design/reconstruction-impact/input-views.md#gene-space-presentation)

### 1.5 Analysis-specific preprocessing

**Method:** [Partition complexity](../../../docs/design/reconstruction-impact/impact-analysis.md#partition-complexity); [Moran method](../../../docs/design/reconstruction-impact/gene-and-function.md#moran); [EMT score](../../../docs/design/reconstruction-impact/gene-and-function.md#emt-score)

In [ ]:
display(pd.DataFrame([{"analysis": "partition", **CONFIG["partition_change"]}, {"analysis": "Moran", "normalization": "normalize_total(1e4), log1p", "min_units": 51}, {"analysis": "AUCell", "resource": "Hallmark 2025.1 EMT", "AUC_threshold": 0.01, "seed": SEED}]))
figures.save_json({"sample": SAMPLE_ID, "route": CONFIG["route_kind"], "seed": SEED, "partition": CONFIG["partition_change"], "spatial": CONFIG["spatial_region"]}, "parameters.json")

<!-- impact-node:1.5 -->
**Analysis-specific preprocessing — batch interpretation**

Method: [input-views.md#preprocessing-presentation](../../../docs/design/reconstruction-impact/input-views.md#preprocessing-presentation)

<!-- impact-node:1.6 -->
**Foundation conclusion — batch interpretation**

Method: [outputs-and-test-plan.md](../../../docs/design/reconstruction-impact/outputs-and-test-plan.md)

## 2. Overall changes — what changed?

**Method:** [Partition complexity](../../../docs/design/reconstruction-impact/impact-analysis.md#partition-complexity); [matched-K](../../../docs/design/reconstruction-impact/impact-analysis.md#matched-k)

### 2.1 Partition complexity diagnostic

In [ ]:
from revise.analysis.reconstruction_impact import run_partition_analysis
PARTITIONS = {}
if CONFIG["route_kind"] == "sp_svc":
    raw_global, recon_global = GLOBAL_INPUTS
    GLOBAL_PARTITION = run_partition_analysis(
        raw_global, recon_global, level1_col=LEVEL1, route_kind="sp_svc",
        resolution_mode="level1_ari", resolution_candidates=CONFIG["partition_change"]["level1_resolution_candidates"],
        within_level1_resolution=CONFIG["partition_change"]["within_level1_resolution"], random_state=SEED,
        n_top_genes=CONFIG["partition_change"]["n_top_genes"],
        raw_qc_min_genes=CONFIG["partition_change"]["raw_qc_min_genes"], raw_qc_min_cells=CONFIG["partition_change"]["raw_qc_min_cells"],
    )
    PARTITIONS["All"] = GLOBAL_PARTITION
for parent, run in RUNS.items():
    raw = run["raw_expression"]
    spatial = run.get("spatial_carrier", run["recon_expression"])
    partition = run_partition_analysis(
        raw, spatial, level1_col=LEVEL1, route_kind=CONFIG["route_kind"],
        final_cluster_key="SVC_cluster" if CONFIG["route_kind"] == "sc_svc" else None,
        resolution_mode="fixed_within_level1", within_level1_resolution=CONFIG["partition_change"]["within_level1_resolution"],
        random_state=SEED, n_top_genes=CONFIG["partition_change"]["n_top_genes"],
        raw_qc_min_genes=CONFIG["partition_change"].get("raw_qc_min_genes", 50),
        raw_qc_min_cells=CONFIG["partition_change"].get("raw_qc_min_cells", 3),
    )
    comparison = next(iter(partition.comparisons.values()))
    ids = comparison.assignments.index
    run.update(partition=partition, comparison=comparison, used=len(ids))
    run["partition_coordinates"] = coordinates(raw).loc[ids] * CONFIG["spatial_region"]["microns_per_coordinate"]
    PARTITIONS[parent] = partition
COMP = {scope: next(iter(partition.comparisons.values())) for scope, partition in PARTITIONS.items()}
COMP_STATUS = {scope: partition.matched_cluster_status for scope, partition in PARTITIONS.items()}
display(pd.DataFrame([{"scope": scope, "Raw-QC retained units": len(COMP[scope].assignments), "n_features": len(part.feature_names), "representation_audit": part.representation_audit, **part.audit} for scope, part in PARTITIONS.items()]))

In [ ]:
complexity_table = pd.DataFrame([
    {"scope": scope, "Raw K": c.summary.iloc[0].n_raw_clusters, "Recon K at Raw resolution": c.summary.iloc[0].n_recon_clusters, "ARI": c.summary.iloc[0]["ARI"]}
    for scope, part in PARTITIONS.items() for c in part.complexity_comparisons.values()
])
display(complexity_table.round(4))
figures.save_table(complexity_table, "reconstruction_impact/partition/complexity_summary.csv")
display(Markdown("Cluster counts describe representation complexity; they are not assignment-change fractions."))
for scope, part in PARTITIONS.items():
    display(part.complexity_sweep)
    display(part.sweep)

<!-- impact-node:2.1 -->
**Partition complexity — batch interpretation**

Method: [impact-analysis.md#partition-complexity](../../../docs/design/reconstruction-impact/impact-analysis.md#partition-complexity)

### 2.2 Matched-complexity change

**Method:** [Matched-K method](../../../docs/design/reconstruction-impact/impact-analysis.md#matched-k)

In [ ]:
plot_comp = {(scope if COMP_STATUS[scope] != "unmatched_cluster_complexity" else scope + " [audit: unmatched K]"): c for scope, c in COMP.items()}
fig = figures.plot_contingency(plot_comp, "Cluster-count-controlled contingency")
figures.save_figure(fig, "matched_k_contingency")
plt.show()
change_table = pd.DataFrame([
    {"scope": scope, "paired units": c.summary.iloc[0].n_units, "Raw K": c.summary.iloc[0].n_raw_clusters,
     "Recon K": c.summary.iloc[0].n_recon_clusters, "ST-unit change": c.summary.iloc[0].st_unit_change_fraction,
     "balanced change": c.summary.iloc[0].balanced_cluster_change, "ARI": c.summary.iloc[0]["ARI"],
     "status": COMP_STATUS[scope], "headline_eligible": COMP_STATUS[scope] != "unmatched_cluster_complexity"}
    for scope, c in COMP.items()
])
display(change_table.round(4))
figures.save_table(change_table, "reconstruction_impact/partition/matched_k_summary.csv")

<!-- impact-node:2.2 -->
**Assignment change after complexity control — batch interpretation**

Method: [impact-analysis.md#matched-k](../../../docs/design/reconstruction-impact/impact-analysis.md#matched-k)

### 2.3 Change by cell type

**Method:** [Changed units and assignment basis](../../../docs/design/reconstruction-impact/impact-analysis.md#changed-units)

In [ ]:
level1_change = pd.concat([part.change_by_level1.assign(scope=scope) for scope, part in PARTITIONS.items()], ignore_index=True)
fig, ax = plt.subplots(figsize=(7, 5.8))
if "All" in PARTITIONS:
    plot_table = level1_change.loc[level1_change.scope.eq("All") & ~level1_change.level1.eq("Overall")].sort_values("change_fraction")
    eligible = COMP_STATUS["All"] != "unmatched_cluster_complexity"
else:
    plot_table = level1_change.drop_duplicates("scope").copy()
    eligible = True
if eligible:
    low = plot_table.change_fraction - plot_table.wilson_ci_lower
    high = plot_table.wilson_ci_upper - plot_table.change_fraction
    labels = plot_table.level1 if "All" in PARTITIONS else plot_table.scope
    ax.barh(labels, plot_table.change_fraction, xerr=np.vstack([low, high]), color="#4c78a8", capsize=3)
    ax.set(xlabel="changed fraction", title="Global Level1 change" if "All" in PARTITIONS else "Parent-internal reassignment")
    if "All" not in PARTITIONS:
        ax.set_yticks(range(len(labels)), [label + (" [audit: unmatched K]" if COMP_STATUS[label] == "unmatched_cluster_complexity" else "") for label in labels])
else:
    ax.axis("off")
    ax.text(.5, .5, "No matched-K headline estimate: unmatched cluster complexity", ha="center", wrap=True)
fig.tight_layout()
figures.save_figure(fig, "level1_change_fraction")
plt.show()
display(level1_change.round(4))
figures.save_table(level1_change, "reconstruction_impact/partition/level1_summary.csv")

In [ ]:
COMPARISONS = save_partition_evidence(
    PARTITIONS, GLOBAL_INPUTS, RUNS, SAMPLE_ID, SEED, figures,
    complexity_kind=("raw_reference_vs_final"
                     if CONFIG["route_kind"] == "sc_svc"
                     else "same_resolution"),
)

<!-- impact-node:2.3 -->
**Which cell types show change — batch interpretation**

Method: [impact-analysis.md#matched-k](../../../docs/design/reconstruction-impact/impact-analysis.md#matched-k)

### 2.4 Gene spatial autocorrelation: Moran I

**Method:** [Moran method](../../../docs/design/reconstruction-impact/gene-and-function.md#moran)

In [ ]:
# Prepare and audit exact paired IDs before normalization.
from reproduce.case.reconstruction_impact.notebook_analysis import prepare_moran
MORAN_MIN_UNITS, MORAN_N_NEIGHS = 51, 6
MORAN_SCOPE_ORDER = []  # Keep the preparation stage visible in the notebook.
MORAN_SOURCES = {}
MORAN_COORDINATE_SOURCES = {}
(
    MORAN_SCOPE_ORDER, MORAN_SOURCES, MORAN_COORDINATE_SOURCES,
    _coordinate_unit, _coordinate_to_microns, MORAN_INPUT_AUDIT,
) = prepare_moran(
    RUNS, CONFIG,
    global_partition=globals().get("GLOBAL_PARTITION"),
    raw_global=globals().get("raw_global"), recon_global=globals().get("recon_global"),
    moran_min_units=MORAN_MIN_UNITS, moran_n_neighs=MORAN_N_NEIGHS,
)
display(MORAN_INPUT_AUDIT)

In [ ]:
# Normalize full-gene views and compute Moran on the same graph within each scope.
from reproduce.case.reconstruction_impact.notebook_analysis import compute_moran
MORAN, MORAN_GRAPH_AUDIT = compute_moran(
    MORAN_SOURCES, MORAN_SCOPE_ORDER, RUNS, CONFIG, OUTPUT_DIR,
    MORAN_COORDINATE_SOURCES, _coordinate_unit, _coordinate_to_microns,
    MORAN_MIN_UNITS=MORAN_MIN_UNITS, MORAN_N_NEIGHS=MORAN_N_NEIGHS,
)
display(MORAN_GRAPH_AUDIT)
display(MORAN.head())


In [ ]:
# Summarize the complete gene union; preserve every status and denominator.
from reproduce.case.reconstruction_impact.notebook_analysis import summarize_moran
MORAN_GENE_AVAILABILITY, MORAN_SUMMARY, MORAN_DISPLAY_SUMMARY = summarize_moran(
    MORAN, MORAN_SOURCES, MORAN_SCOPE_ORDER
)
figures.save_table(MORAN_GENE_AVAILABILITY, "gene_availability.csv", scope="moran")
figures.save_table(MORAN_SUMMARY, "moran_summary.csv", scope="moran")
figures.save_table(MORAN_DISPLAY_SUMMARY, "moran_distribution_summary.csv", scope="moran")
display(MORAN_SUMMARY)
display(MORAN_DISPLAY_SUMMARY)
display(MORAN_GENE_AVAILABILITY.groupby(
    ["scope", "raw_status", "reconstruction_status"], dropna=False
).size().rename("n_genes").reset_index())

In [ ]:
# Compare all-valid and shared-valid Q75 summaries using one color scale.
fig = figures.plot_moran_q75(MORAN_SUMMARY, MORAN_SCOPE_ORDER)
figures.save_figure(fig, "moran_q75_heatmap")
plt.show()
display(MORAN_DISPLAY_SUMMARY.loc[:, ["scope", "gene_set", "side", "n_valid", "median", "q1", "q75"]])

In [ ]:
# Display all computed genes and the paired shared-gene subset.
from reproduce.case.reconstruction_impact.notebook_analysis import build_moran_distribution
MORAN_DISTRIBUTION = build_moran_distribution(MORAN, MORAN_SCOPE_ORDER)
fig = figures.plot_moran_distribution(MORAN_DISTRIBUTION, MORAN_SCOPE_ORDER)
figures.save_figure(fig, "moran_all_gene_distribution")
plt.show()
display(MORAN_DISPLAY_SUMMARY.loc[:, ["scope", "gene_set", "side", "n_valid", "median", "q1", "q75"]])

In [ ]:
# Shared-gene comparison remains a descriptive paired view.
fig = figures.plot_moran_shared_scatter(MORAN, MORAN_SCOPE_ORDER)
figures.save_figure(fig, "moran_shared_gene_scatter")
plt.show()

In [ ]:
# Paired deltas complement the side-specific full-gene distributions.
from reproduce.case.reconstruction_impact.notebook_analysis import build_moran_paired_delta
MORAN_PAIRED_DELTA = build_moran_paired_delta(MORAN, MORAN_SCOPE_ORDER)
fig = figures.plot_moran_paired_delta(MORAN_PAIRED_DELTA, MORAN_SCOPE_ORDER)
figures.save_figure(fig, "moran_shared_gene_delta_distribution")
plt.show()
display(MORAN_SUMMARY.loc[:, ["scope", "paired_delta_n", "paired_delta_median", "paired_delta_q1", "paired_delta_q3"]])

<!-- impact-node:2.4 -->
**Full-gene spatial autocorrelation: Moran — batch interpretation**

Method: [gene-and-function.md#moran](../../../docs/design/reconstruction-impact/gene-and-function.md#moran)

### 2.5 Pre-specified EMT sentinel: AUCell

**Method:** [AUCell method](../../../docs/design/reconstruction-impact/gene-and-function.md#emt-coverage) and [score contract](../../../docs/design/reconstruction-impact/gene-and-function.md#emt-score)

In [ ]:
# Score complete expression views with the fixed EMT resource; publish each audit.
from reproduce.case.reconstruction_impact.notebook_analysis import compute_emt
AUCELL_AUC_THRESHOLD = 0.01  # Quantile of detected genes, not a fixed rank fraction.
AUCELL_SEED = 42
AUCELL, AVAIL, aucell_summary, EMT_NAME, GMT_PATH = compute_emt(
    RUNS, ROOT, OUTPUT_DIR, CONFIG,
    AUCELL_AUC_THRESHOLD=AUCELL_AUC_THRESHOLD, AUCELL_SEED=AUCELL_SEED,
)
AUCELL_AVAIL = AVAIL
display(aucell_summary.round(4))


#### AUCell resource coverage and computability

In [ ]:
# Display the already-published coverage audit; do not rerun AUCell.
_AU_SCOPE_LABELS = {
    "Fibroblast": "Fibroblast",
    "Mono_Macro": "Mono/Macro",
    "Mono/Macro": "Mono/Macro",
    "T": "T",
}
_AU_SCOPE_ORDER = [
    scope for scope in ("Fibroblast", "Mono_Macro", "Mono/Macro", "T")
    if scope in set(AVAIL.get("scope", pd.Series(dtype=object)).astype(str))
]
_AU_COVERAGE = AVAIL.copy()
if not _AU_COVERAGE.empty:
    _AU_COVERAGE["scope"] = _AU_COVERAGE["scope"].astype(str)
    _AU_COVERAGE["scope_label"] = _AU_COVERAGE["scope"].map(_AU_SCOPE_LABELS).fillna(_AU_COVERAGE["scope"])
    _AU_COVERAGE["scope_order"] = _AU_COVERAGE["scope"].map({value: index for index, value in enumerate(_AU_SCOPE_ORDER)})
    _AU_COVERAGE = _AU_COVERAGE.sort_values(["scope_order", "side"], kind="stable")
    display(_AU_COVERAGE.loc[:, [
        "scope_label", "side", "pathway", "resource_gene_count",
        "available_gene_count", "coverage", "detected_signature_gene_count",
        "n_observations", "n_genes", "detected_count_quantile",
        "provider_auc_threshold", "effective_rank_length",
        "rank_cutoff_zero_based", "status", "reason",
    ]].reset_index(drop=True).round(4))
else:
    display(Markdown("No AUCell coverage rows were published."))


#### Overall AUCell distributions and paired change

In [ ]:
# Display published AUCell scores; the scorer is called in the prior cell.
_AU_LONG, _AU_DELTA, _AU_SUMMARY, _AU_COMPUTED = prepare_aucell_overall(
    AUCELL, aucell_summary, _AU_SCOPE_LABELS, _AU_SCOPE_ORDER
)
_AU_FIG = figures.plot_aucell_overall(_AU_LONG, _AU_DELTA, _AU_SCOPE_ORDER, _AU_SCOPE_LABELS, EMT_NAME)
figures.save_figure(_AU_FIG, "aucell_overall_distribution_and_delta")
plt.show()
_AU_SUMMARY_COLUMNS = [
    "scope", "pathway", "raw_status", "reconstruction_status", "comparison_status",
    "raw_n_valid", "raw_median", "raw_q1", "raw_q3", "reconstruction_n_valid",
    "reconstruction_median", "reconstruction_q1", "reconstruction_q3", "paired_n",
    "paired_delta_median", "paired_delta_q1", "paired_delta_q3",
]
if _AU_SUMMARY.empty:
    display(Markdown("No AUCell summary rows were published."))
else:
    display(_AU_SUMMARY.loc[:, [c for c in _AU_SUMMARY_COLUMNS if c in _AU_SUMMARY.columns]].reset_index(drop=True).round(4))


#### 1. Fibroblast: EMT sentinel score comparison

In [ ]:
# Parent display only: all values come from AUCELL/aucell_summary above.
_AU_PARENT_SCOPE = 'Fibroblast'
_AU_PARENT_LABEL = 'Fibroblast'
_AU_PARENT, _AU_PARENT_LONG, _AU_PARENT_DELTA, _AU_PARENT_SUMMARY, _ = prepare_aucell_parent(
    AUCELL, aucell_summary, _AU_PARENT_SCOPE, _AU_PARENT_LABEL
)
_AU_PARENT_FIG = figures.plot_aucell_parent(_AU_PARENT_LONG, _AU_PARENT_DELTA, _AU_PARENT_LABEL, EMT_NAME)
figures.save_figure(_AU_PARENT_FIG, f"aucell_{_AU_PARENT_SCOPE.lower().replace('/', '_')}_distribution_and_delta")
plt.show()
if _AU_PARENT_SUMMARY.empty:
    display(Markdown("No AUCell summary row was published."))
else:
    display(_AU_PARENT_SUMMARY.round(4))


#### 2. Mono/Macro: EMT sentinel score comparison

In [ ]:
# Parent display only: all values come from AUCELL/aucell_summary above.
_AU_PARENT_SCOPE = 'Mono_Macro'
_AU_PARENT_LABEL = 'Mono/Macro'
_AU_PARENT, _AU_PARENT_LONG, _AU_PARENT_DELTA, _AU_PARENT_SUMMARY, _ = prepare_aucell_parent(
    AUCELL, aucell_summary, _AU_PARENT_SCOPE, _AU_PARENT_LABEL
)
_AU_PARENT_FIG = figures.plot_aucell_parent(_AU_PARENT_LONG, _AU_PARENT_DELTA, _AU_PARENT_LABEL, EMT_NAME)
figures.save_figure(_AU_PARENT_FIG, f"aucell_{_AU_PARENT_SCOPE.lower().replace('/', '_')}_distribution_and_delta")
plt.show()
if _AU_PARENT_SUMMARY.empty:
    display(Markdown("No AUCell summary row was published."))
else:
    display(_AU_PARENT_SUMMARY.round(4))


#### 3. T: EMT sentinel score comparison

In [ ]:
# Parent display only: all values come from AUCELL/aucell_summary above.
_AU_PARENT_SCOPE = 'T'
_AU_PARENT_LABEL = 'T'
_AU_PARENT, _AU_PARENT_LONG, _AU_PARENT_DELTA, _AU_PARENT_SUMMARY, _ = prepare_aucell_parent(
    AUCELL, aucell_summary, _AU_PARENT_SCOPE, _AU_PARENT_LABEL
)
_AU_PARENT_FIG = figures.plot_aucell_parent(_AU_PARENT_LONG, _AU_PARENT_DELTA, _AU_PARENT_LABEL, EMT_NAME)
figures.save_figure(_AU_PARENT_FIG, f"aucell_{_AU_PARENT_SCOPE.lower().replace('/', '_')}_distribution_and_delta")
plt.show()
if _AU_PARENT_SUMMARY.empty:
    display(Markdown("No AUCell summary row was published."))
else:
    display(_AU_PARENT_SUMMARY.round(4))


<!-- impact-node:2.5 -->
**EMT: resource coverage and scores — batch interpretation**

Method: [gene-and-function.md#emt-coverage](../../../docs/design/reconstruction-impact/gene-and-function.md#emt-coverage)

### 2.6 Summary of observed changes

**Method:** [Changed units](../../../docs/design/reconstruction-impact/impact-analysis.md#changed-units); [Moran method](../../../docs/design/reconstruction-impact/gene-and-function.md#moran); [EMT score](../../../docs/design/reconstruction-impact/gene-and-function.md#emt-score)

In [ ]:
display(change_table.round(4))
display(MORAN_SUMMARY.round(4))
display(aucell_summary.round(4))
display(Markdown("Next inspect assignment changes and the EMT sentinel field, then local diversity under both Raw baselines."))

<!-- impact-node:2.6 -->
**Overall conclusion — batch interpretation**

Method: [outputs-and-test-plan.md](../../../docs/design/reconstruction-impact/outputs-and-test-plan.md)

## 3. Spatial localization — where are the observed changes?

**Method:** [Anatomy method](../../../docs/design/reconstruction-impact/impact-analysis.md#anatomy); [changed units](../../../docs/design/reconstruction-impact/impact-analysis.md#changed-units)

### 3.1 Level1 anatomy context

In [ ]:
from revise.analysis.reconstruction_impact import compute_anatomy_regions, _map_to_anatomy_regions, _summarize_parent_window_anatomy
from revise.analysis.basic.spatial_region import (
    assign_square_windows,
    compute_rarefied_window_diversity,
    convert_coordinates_to_microns,
    select_region_threshold,
    select_window_scale,
    summarize_region_extent_by_anatomy,
)
ANALYSIS_ROOT = OUTPUT_DIR / "analysis"
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)
ANATOMY = compute_anatomy_regions(
    full_coordinates=full_coordinates, full_level1_labels=full_level1,
    microns_per_coordinate=CONFIG["spatial_region"]["microns_per_coordinate"],
    candidate_window_sides_um=CONFIG["spatial_region"]["candidate_window_sides_um"],
    min_parent_units=4, cell_equivalent_um=CELL_EQUIVALENT_UM,
    **CONFIG["spatial_region"]["anatomy_region"],
)
anatomy_units = save_anatomy_evidence(
    ANATOMY, full_level1, CONFIG, ANALYSIS_ROOT, figures,
)
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
figures.anatomy_map(axes[0, 0], ANATOMY); figures.anatomy_map(axes[0, 1], ANATOMY, "Tumor")
figures.anatomy_map(axes[1, 0], ANATOMY, "Normal"); figures.anatomy_map(axes[1, 1], ANATOMY, "Interface")
fig.suptitle("Level1 anatomy Regions"); fig.tight_layout()
figures.save_figure(fig, "anatomy_regions"); plt.show()
display(ANATOMY.anatomy_context_summary.loc[lambda x: x.level1_region.isin(["Tumor", "Normal", "Interface"])])

#### Anatomy window support

**Method:** [Window support method](../../../docs/design/reconstruction-impact/impact-analysis.md#window-support)

In [ ]:
fig = figures.plot_support_curve(ANATOMY, "Anatomy window support selection")
figures.save_figure(fig, "anatomy_window_decision")
plt.show()
display(ANATOMY.support_sensitivity)

<!-- impact-node:3.1 -->
**Anatomy context — batch interpretation**

Method: [impact-analysis.md#anatomy](../../../docs/design/reconstruction-impact/impact-analysis.md#anatomy)

### 3.2 Changed-unit spatial localization

**Method:** [Changed units](../../../docs/design/reconstruction-impact/impact-analysis.md#changed-units)

In [ ]:
for parent, run in RUNS.items():
    assignments = run["comparison"].assignments.set_index("unit_id")
    raw = run["raw_expression"][assignments.index].copy()
    xy = coordinates(raw) * CONFIG["spatial_region"]["microns_per_coordinate"]
    changed_dir = ANALYSIS_ROOT / "changed_units" / parent
    changed_dir.mkdir(parents=True, exist_ok=True)
    assignments.assign(x=xy["x"], y=xy["y"]).reset_index(names="unit_id").to_csv(changed_dir / "matched_k_assignments.csv.gz", index=False, compression={"method": "gzip", "mtime": 0})
fig = figures.plot_changed_units(RUNS, "Changed-unit spatial localization")
figures.save_figure(fig, "changed_units"); plt.show()
display(change_table[["scope", "status", "headline_eligible"]])
display(Markdown("Unmatched-K maps are diagnostic reassignment locations, not cluster-count-controlled change estimates."))

<!-- impact-node:3.2 -->
**Spatial location of assignment change — batch interpretation**

Method: [impact-analysis.md#changed-units](../../../docs/design/reconstruction-impact/impact-analysis.md#changed-units)

### 3.3 EMT sentinel spatial field

**Method:** [EMT spatial fields](../../../docs/design/reconstruction-impact/gene-and-function.md#emt-spatial-fields)

In [ ]:
FEATURES = save_spatial_feature_values(
    RUNS, AUCELL, EMT_NAME, CONFIG, ANALYSIS_ROOT,
)

In [ ]:
parent = 'Fibroblast'
feature = 'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION'
field = FEATURES.loc[FEATURES.scope.eq(parent) & FEATURES.feature.eq(feature)]
fig = figures.plot_feature_field(field, feature, f"{parent}: {feature} spatial field")
figures.save_figure(fig, f"{parent.lower()}_{feature.lower()}_spatial")
plt.show()
display(field[["raw_value", "reconstruction_value", "delta"]].agg(["count", "median", "mean"]).T)
display(field[["raw_status", "reconstruction_status", "expression_layer"]].drop_duplicates())
display(Markdown("Inspect the EMT score against the same tissue coordinates; unavailable scores remain unavailable."))

In [ ]:
parent = 'Mono_Macro'
feature = 'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION'
field = FEATURES.loc[FEATURES.scope.eq(parent) & FEATURES.feature.eq(feature)]
fig = figures.plot_feature_field(field, feature, f"{parent}: {feature} spatial field")
figures.save_figure(fig, f"{parent.lower()}_{feature.lower()}_spatial")
plt.show()
display(field[["raw_value", "reconstruction_value", "delta"]].agg(["count", "median", "mean"]).T)
display(field[["raw_status", "reconstruction_status", "expression_layer"]].drop_duplicates())
display(Markdown("Inspect the EMT score against the same tissue coordinates; unavailable scores remain unavailable."))

In [ ]:
parent = 'T'
feature = 'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION'
field = FEATURES.loc[FEATURES.scope.eq(parent) & FEATURES.feature.eq(feature)]
fig = figures.plot_feature_field(field, feature, f"{parent}: {feature} spatial field")
figures.save_figure(fig, f"{parent.lower()}_{feature.lower()}_spatial")
plt.show()
display(field[["raw_value", "reconstruction_value", "delta"]].agg(["count", "median", "mean"]).T)
display(field[["raw_status", "reconstruction_status", "expression_layer"]].drop_duplicates())
display(Markdown("Inspect the EMT score against the same tissue coordinates; unavailable scores remain unavailable."))

<!-- impact-node:3.3 -->
**Spatial location of EMT — batch interpretation**

Method: [gene-and-function.md#emt-spatial-fields](../../../docs/design/reconstruction-impact/gene-and-function.md#emt-spatial-fields)

### 3.4 Internal-state baselines and window decision

**Method:** [Local diversity and window definition](../../../docs/design/reconstruction-impact/impact-analysis.md#local-diversity); [window support](../../../docs/design/reconstruction-impact/impact-analysis.md#window-support)

In [ ]:
for parent, run in RUNS.items():
    prepare_local_state(
        parent, run, CONFIG, ROOT, ANATOMY, ANALYSIS_ROOT, figures,
        CELL_EQUIVALENT_UM,
    )

In [ ]:
RUN = RUNS["Fibroblast"]; IMPACT = RUN["impact"]
fig = figures.plot_support_curve(IMPACT, "Fibroblast: occupancy-only window decision")
figures.save_figure(fig, "fibroblast_window_decision"); plt.show()
occupancy = IMPACT.unit_assignments.groupby("window_id").size()
sampled = RUN.get("sampled", RUN["used"])
decision = pd.DataFrame([{"available units": RUN["available"], "sampled units": sampled, "Raw-QC retained units": RUN["used"], "sampling mode": "full" if RUN["available"] == sampled else "deterministic 30k", "seed": CONFIG["partition_change"]["random_state"], "window side (cells per side)": IMPACT.scale_audit["main_window_cells_per_side"], "window side (µm)": IMPACT.scale_audit["main_window_side_um"], "occupancy median [Q1,Q3]": compact_iqr(occupancy), "minimum support": IMPACT.scale_audit["min_parent_units"], "rarefaction units": IMPACT.scale_audit["min_parent_units"], "paired draws": IMPACT.scale_audit["rarefaction_draws"]}])
display(decision)

In [ ]:
METRICS = RUN["impact"].window_metrics.copy(); METRICS.attrs["side_um"] = IMPACT.scale_audit["main_window_side_um"]
fig = figures.plot_reconstructed_clusters(RUN, "Fibroblast: reconstructed internal state")
figures.save_figure(fig, "fibroblast_reconstructed_clusters"); plt.show()
display(pd.DataFrame([{"baseline": "Raw Level1 (uniform parent)", "Kobs": 1.0, "Neff": 1.0, "evenness": 1.0}, {"baseline": "Raw Level2 mapping", "Kobs": RUN["raw_level2"].audit["n_mapped_level2"], "Neff": "window-specific", "evenness": "window-specific"}]))
display(pd.DataFrame([RUN["raw_level2"].audit]).loc[:, ["source", "method", "n_raw_units", "n_reference_cells", "n_reference_level2", "n_mapped_level2", "n_missing"]])

In [ ]:
fig = figures.plot_cluster_pair(RUN, "Fibroblast: matched-K cluster assignment")
figures.save_figure(fig, "fibroblast_matched_clusters"); plt.show()
row = RUN["comparison"].summary.iloc[0]
display(pd.DataFrame([{"paired units": row.n_units, "Raw K": row.n_raw_clusters, "Recon K": row.n_recon_clusters, "parent-internal reassignment": row.st_unit_change_fraction, "balanced change": row.balanced_cluster_change, "ARI": row["ARI"]}]).round(4))

In [ ]:
RUN = RUNS["Mono_Macro"]; IMPACT = RUN["impact"]
fig = figures.plot_support_curve(IMPACT, "Mono_Macro: occupancy-only window decision")
figures.save_figure(fig, "mono_macro_window_decision"); plt.show()
occupancy = IMPACT.unit_assignments.groupby("window_id").size()
sampled = RUN.get("sampled", RUN["used"])
decision = pd.DataFrame([{"available units": RUN["available"], "sampled units": sampled, "Raw-QC retained units": RUN["used"], "sampling mode": "full" if RUN["available"] == sampled else "deterministic 30k", "seed": CONFIG["partition_change"]["random_state"], "window side (cells per side)": IMPACT.scale_audit["main_window_cells_per_side"], "window side (µm)": IMPACT.scale_audit["main_window_side_um"], "occupancy median [Q1,Q3]": compact_iqr(occupancy), "minimum support": IMPACT.scale_audit["min_parent_units"], "rarefaction units": IMPACT.scale_audit["min_parent_units"], "paired draws": IMPACT.scale_audit["rarefaction_draws"]}])
display(decision)

In [ ]:
METRICS = RUN["impact"].window_metrics.copy(); METRICS.attrs["side_um"] = IMPACT.scale_audit["main_window_side_um"]
fig = figures.plot_reconstructed_clusters(RUN, "Mono_Macro: reconstructed internal state")
figures.save_figure(fig, "mono_macro_reconstructed_clusters"); plt.show()
display(pd.DataFrame([{"baseline": "Raw Level1 (uniform parent)", "Kobs": 1.0, "Neff": 1.0, "evenness": 1.0}, {"baseline": "Raw Level2 mapping", "Kobs": RUN["raw_level2"].audit["n_mapped_level2"], "Neff": "window-specific", "evenness": "window-specific"}]))
display(pd.DataFrame([RUN["raw_level2"].audit]).loc[:, ["source", "method", "n_raw_units", "n_reference_cells", "n_reference_level2", "n_mapped_level2", "n_missing"]])

In [ ]:
fig = figures.plot_cluster_pair(RUN, "Mono_Macro: matched-K cluster assignment")
figures.save_figure(fig, "mono_macro_matched_clusters"); plt.show()
row = RUN["comparison"].summary.iloc[0]
display(pd.DataFrame([{"paired units": row.n_units, "Raw K": row.n_raw_clusters, "Recon K": row.n_recon_clusters, "parent-internal reassignment": row.st_unit_change_fraction, "balanced change": row.balanced_cluster_change, "ARI": row["ARI"]}]).round(4))

In [ ]:
RUN = RUNS["T"]; IMPACT = RUN["impact"]
fig = figures.plot_support_curve(IMPACT, "T: occupancy-only window decision")
figures.save_figure(fig, "t_window_decision"); plt.show()
occupancy = IMPACT.unit_assignments.groupby("window_id").size()
sampled = RUN.get("sampled", RUN["used"])
decision = pd.DataFrame([{"available units": RUN["available"], "sampled units": sampled, "Raw-QC retained units": RUN["used"], "sampling mode": "full" if RUN["available"] == sampled else "deterministic 30k", "seed": CONFIG["partition_change"]["random_state"], "window side (cells per side)": IMPACT.scale_audit["main_window_cells_per_side"], "window side (µm)": IMPACT.scale_audit["main_window_side_um"], "occupancy median [Q1,Q3]": compact_iqr(occupancy), "minimum support": IMPACT.scale_audit["min_parent_units"], "rarefaction units": IMPACT.scale_audit["min_parent_units"], "paired draws": IMPACT.scale_audit["rarefaction_draws"]}])
display(decision)

In [ ]:
METRICS = RUN["impact"].window_metrics.copy(); METRICS.attrs["side_um"] = IMPACT.scale_audit["main_window_side_um"]
fig = figures.plot_reconstructed_clusters(RUN, "T: reconstructed internal state")
figures.save_figure(fig, "t_reconstructed_clusters"); plt.show()
display(pd.DataFrame([{"baseline": "Raw Level1 (uniform parent)", "Kobs": 1.0, "Neff": 1.0, "evenness": 1.0}, {"baseline": "Raw Level2 mapping", "Kobs": RUN["raw_level2"].audit["n_mapped_level2"], "Neff": "window-specific", "evenness": "window-specific"}]))
display(pd.DataFrame([RUN["raw_level2"].audit]).loc[:, ["source", "method", "n_raw_units", "n_reference_cells", "n_reference_level2", "n_mapped_level2", "n_missing"]])

In [ ]:
fig = figures.plot_cluster_pair(RUN, "T: matched-K cluster assignment")
figures.save_figure(fig, "t_matched_clusters"); plt.show()
row = RUN["comparison"].summary.iloc[0]
display(pd.DataFrame([{"paired units": row.n_units, "Raw K": row.n_raw_clusters, "Recon K": row.n_recon_clusters, "parent-internal reassignment": row.st_unit_change_fraction, "balanced change": row.balanced_cluster_change, "ARI": row["ARI"]}]).round(4))

<!-- impact-node:3.4 -->
**Foundation for local-state comparison — batch interpretation**

Method: [impact-analysis.md#window-support](../../../docs/design/reconstruction-impact/impact-analysis.md#window-support)

### 3.5 Local richness: Kobs

**Method:** [Local diversity method](../../../docs/design/reconstruction-impact/impact-analysis.md#local-diversity)

In [ ]:
for parent, run in RUNS.items():
    spatial = run["spatial"]
    metrics = compute_rarefied_window_diversity(spatial["windows"], spatial["assignments"]["raw_cluster"], spatial["assignments"]["recon_cluster"], raw_level2_labels=spatial["level2"].labels, min_parent_units=4, n_draws=200, random_state=SEED)
    metrics["scope"] = parent
    metrics["scale_um"] = spatial["side_um"]
    run["spatial"]["metrics"] = metrics
    run["impact"].window_metrics = metrics
    metrics.to_csv(ANALYSIS_ROOT / "local_state" / parent / "window_metrics.csv", index=False)

In [ ]:
RUN = RUNS['Fibroblast']; IMPACT = RUN["impact"]; METRICS = IMPACT.window_metrics
fig = figures.plot_metric_comparison(RUN, "k_obs", "Fibroblast: Kobs evidence matrix")
figures.save_figure(fig, "fibroblast_kobs_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["k_obs_raw"]), "Recon minus baseline": compact_iqr(valid["delta_k_obs_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["k_obs_level2"]), "Recon minus baseline": compact_iqr(valid["delta_k_obs_vs_raw_level2"])}]))
display(valid[["k_obs_recon", "delta_k_obs_vs_raw_leiden", "delta_k_obs_vs_raw_level2"]].agg(["count", "median", "mean"]).T)

In [ ]:
RUN = RUNS['Mono_Macro']; IMPACT = RUN["impact"]; METRICS = IMPACT.window_metrics
fig = figures.plot_metric_comparison(RUN, "k_obs", "Mono_Macro: Kobs evidence matrix")
figures.save_figure(fig, "mono_macro_kobs_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["k_obs_raw"]), "Recon minus baseline": compact_iqr(valid["delta_k_obs_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["k_obs_level2"]), "Recon minus baseline": compact_iqr(valid["delta_k_obs_vs_raw_level2"])}]))
display(valid[["k_obs_recon", "delta_k_obs_vs_raw_leiden", "delta_k_obs_vs_raw_level2"]].agg(["count", "median", "mean"]).T)

In [ ]:
RUN = RUNS['T']; IMPACT = RUN["impact"]; METRICS = IMPACT.window_metrics
fig = figures.plot_metric_comparison(RUN, "k_obs", "T: Kobs evidence matrix")
figures.save_figure(fig, "t_kobs_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["k_obs_raw"]), "Recon minus baseline": compact_iqr(valid["delta_k_obs_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["k_obs_level2"]), "Recon minus baseline": compact_iqr(valid["delta_k_obs_vs_raw_level2"])}]))
display(valid[["k_obs_recon", "delta_k_obs_vs_raw_leiden", "delta_k_obs_vs_raw_level2"]].agg(["count", "median", "mean"]).T)

<!-- impact-node:3.5 -->
**Kobs: versus Raw Leiden and Raw Level2 — batch interpretation**

Method: [impact-analysis.md#local-diversity](../../../docs/design/reconstruction-impact/impact-analysis.md#local-diversity)

### 3.6 Effective diversity: Neff

**Method:** [Local diversity method](../../../docs/design/reconstruction-impact/impact-analysis.md#local-diversity)

In [ ]:
RUN = RUNS['Fibroblast']; IMPACT = RUN["impact"]; METRICS = IMPACT.window_metrics
fig = figures.plot_metric_comparison(RUN, "neff", "Fibroblast: Neff evidence matrix")
figures.save_figure(fig, "fibroblast_neff_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["neff_raw"]), "Recon minus baseline": compact_iqr(valid["delta_neff_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["neff_level2"]), "Recon minus baseline": compact_iqr(valid["delta_neff_vs_raw_level2"])}]))
display(valid[["neff_recon", "delta_neff_vs_raw_leiden", "delta_neff_vs_raw_level2"]].agg(["count", "median", "mean"]).T)

In [ ]:
RUN = RUNS['Mono_Macro']; IMPACT = RUN["impact"]; METRICS = IMPACT.window_metrics
fig = figures.plot_metric_comparison(RUN, "neff", "Mono_Macro: Neff evidence matrix")
figures.save_figure(fig, "mono_macro_neff_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["neff_raw"]), "Recon minus baseline": compact_iqr(valid["delta_neff_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["neff_level2"]), "Recon minus baseline": compact_iqr(valid["delta_neff_vs_raw_level2"])}]))
display(valid[["neff_recon", "delta_neff_vs_raw_leiden", "delta_neff_vs_raw_level2"]].agg(["count", "median", "mean"]).T)

In [ ]:
RUN = RUNS['T']; IMPACT = RUN["impact"]; METRICS = IMPACT.window_metrics
fig = figures.plot_metric_comparison(RUN, "neff", "T: Neff evidence matrix")
figures.save_figure(fig, "t_neff_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["neff_raw"]), "Recon minus baseline": compact_iqr(valid["delta_neff_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["neff_level2"]), "Recon minus baseline": compact_iqr(valid["delta_neff_vs_raw_level2"])}]))
display(valid[["neff_recon", "delta_neff_vs_raw_leiden", "delta_neff_vs_raw_level2"]].agg(["count", "median", "mean"]).T)

<!-- impact-node:3.6 -->
**Neff: versus Raw Leiden and Raw Level2 — batch interpretation**

Method: [impact-analysis.md#local-diversity](../../../docs/design/reconstruction-impact/impact-analysis.md#local-diversity)

### 3.7 Evenness

**Method:** [Local diversity method](../../../docs/design/reconstruction-impact/impact-analysis.md#local-diversity)

In [ ]:
RUN = RUNS['Fibroblast']; IMPACT = RUN["impact"]; METRICS = IMPACT.window_metrics
fig = figures.plot_metric_comparison(RUN, "evenness", "Fibroblast: evenness evidence matrix")
figures.save_figure(fig, "fibroblast_evenness_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["evenness_raw"]), "Recon minus baseline": compact_iqr(valid["delta_evenness_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["evenness_level2"]), "Recon minus baseline": compact_iqr(valid["delta_evenness_vs_raw_level2"])}]))
display(valid[["evenness_recon", "delta_evenness_vs_raw_leiden", "delta_evenness_vs_raw_level2"]].agg(["count", "median", "mean"]).T)

In [ ]:
RUN = RUNS['Mono_Macro']; IMPACT = RUN["impact"]; METRICS = IMPACT.window_metrics
fig = figures.plot_metric_comparison(RUN, "evenness", "Mono_Macro: evenness evidence matrix")
figures.save_figure(fig, "mono_macro_evenness_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["evenness_raw"]), "Recon minus baseline": compact_iqr(valid["delta_evenness_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["evenness_level2"]), "Recon minus baseline": compact_iqr(valid["delta_evenness_vs_raw_level2"])}]))
display(valid[["evenness_recon", "delta_evenness_vs_raw_leiden", "delta_evenness_vs_raw_level2"]].agg(["count", "median", "mean"]).T)

In [ ]:
RUN = RUNS['T']; IMPACT = RUN["impact"]; METRICS = IMPACT.window_metrics
fig = figures.plot_metric_comparison(RUN, "evenness", "T: evenness evidence matrix")
figures.save_figure(fig, "t_evenness_matrix"); plt.show()
valid = METRICS.loc[METRICS.valid_window]
display(pd.DataFrame([{"baseline": "Raw Leiden", "state median [Q1,Q3]": compact_iqr(valid["evenness_raw"]), "Recon minus baseline": compact_iqr(valid["delta_evenness_vs_raw_leiden"])}, {"baseline": "Raw Level2", "state median [Q1,Q3]": compact_iqr(valid["evenness_level2"]), "Recon minus baseline": compact_iqr(valid["delta_evenness_vs_raw_level2"])}]))
display(valid[["evenness_recon", "delta_evenness_vs_raw_leiden", "delta_evenness_vs_raw_level2"]].agg(["count", "median", "mean"]).T)

<!-- impact-node:3.7 -->
**Evenness: supporting interpretation — batch interpretation**

Method: [impact-analysis.md#local-diversity](../../../docs/design/reconstruction-impact/impact-analysis.md#local-diversity)

### 3.8 Anatomy-stratified summaries

**Method:** [Anatomy method](../../../docs/design/reconstruction-impact/impact-analysis.md#anatomy)

In [ ]:
for parent, run in RUNS.items():
    change_summary, anatomy_summary = save_local_state_evidence(
        parent, run, ANATOMY, ANALYSIS_ROOT,
    )
    display(change_summary)
    display(anatomy_summary)

<!-- impact-node:3.8 -->
**Anatomy-stratified local results — batch interpretation**

Method: [impact-analysis.md#local-diversity](../../../docs/design/reconstruction-impact/impact-analysis.md#local-diversity)

<!-- impact-node:3.9 -->
**Localization and local-state conclusion — batch interpretation**

Method: [outputs-and-test-plan.md](../../../docs/design/reconstruction-impact/outputs-and-test-plan.md)

## 4. Regions: local-state coverage and reliability

**Method:** [Region reliability](../../../docs/design/reconstruction-impact/impact-analysis.md#region-reliability); [Region extent](../../../docs/design/reconstruction-impact/impact-analysis.md#region-extent)

### 4.1 Threshold reliability — State Region

In [ ]:
for parent, run in RUNS.items():
    spatial = run["spatial"]; metrics = spatial["metrics"].copy()
    valid = metrics.loc[metrics.valid_window]
    state_threshold, bootstrap = select_region_threshold(valid.neff_recon.to_numpy(), n_bootstrap=CONFIG["spatial_region"]["threshold_bootstraps"], random_state=SEED)
    gain_values = valid.loc[valid.delta_neff_vs_raw_leiden.gt(0), "delta_neff_vs_raw_leiden"].to_numpy()
    gain_threshold, gain_bootstrap = select_region_threshold(gain_values, n_bootstrap=500, random_state=SEED)
    metrics["in_state_region"] = pd.Series(pd.NA, index=metrics.index, dtype="boolean")
    metrics["in_gain_region"] = pd.Series(pd.NA, index=metrics.index, dtype="boolean")
    if gain_threshold["status"] == "ok":
        metrics.loc[metrics.valid_window, "in_gain_region"] = metrics.loc[metrics.valid_window, "delta_neff_vs_raw_leiden"].ge(float(gain_threshold["threshold"]))
    if state_threshold["status"] == "ok":
        metrics.loc[metrics.valid_window, "in_state_region"] = metrics.loc[metrics.valid_window, "neff_recon"].ge(float(state_threshold["threshold"]))
    impact = SimpleNamespace(window_metrics=metrics, scale_audit={"main_window_side_um": spatial["side_um"]}, state_threshold=state_threshold)
    run["impact"] = impact
    spatial.update(metrics=metrics, state_threshold=state_threshold, state_bootstrap=bootstrap, gain_threshold=gain_threshold, gain_bootstrap=gain_bootstrap)

In [ ]:
# Diagnose all three parents before showing any Region masks.
THRESHOLD_RELIABILITY, fig = plot_threshold_reliability(
    RUNS, CONFIG["spatial_region"]["threshold_bootstraps"])
figures.save_table(THRESHOLD_RELIABILITY, "threshold_reliability.csv")
figures.save_figure(fig, "threshold_reliability")
plt.show()
display(THRESHOLD_RELIABILITY)

<!-- impact-node:4.1 -->
**Identifiability and threshold reliability — batch interpretation**

Method: [impact-analysis.md#region-reliability](../../../docs/design/reconstruction-impact/impact-analysis.md#region-reliability)

### 4.2 Continuous reconstructed-Neff state and mask

**Method:** [Region extent](../../../docs/design/reconstruction-impact/impact-analysis.md#region-extent)

In [ ]:
run = RUNS['Fibroblast']
fig = figures.plot_high_diversity(run, "Fibroblast: reconstructed Neff and State Region")
figures.save_figure(fig, "fibroblast_state_region")
plt.show()
display(pd.DataFrame([run["impact"].state_threshold]))
display(Markdown("The continuous state and Region mask answer different questions; an unavailable threshold yields no coverage estimate."))

In [ ]:
run = RUNS['Mono_Macro']
fig = figures.plot_high_diversity(run, "Mono_Macro: reconstructed Neff and State Region")
figures.save_figure(fig, "mono_macro_state_region")
plt.show()
display(pd.DataFrame([run["impact"].state_threshold]))
display(Markdown("The continuous state and Region mask answer different questions; an unavailable threshold yields no coverage estimate."))

In [ ]:
run = RUNS['T']
fig = figures.plot_high_diversity(run, "T: reconstructed Neff and State Region")
figures.save_figure(fig, "t_state_region")
plt.show()
display(pd.DataFrame([run["impact"].state_threshold]))
display(Markdown("The continuous state and Region mask answer different questions; an unavailable threshold yields no coverage estimate."))

<!-- impact-node:4.2 -->
**Continuous state and State Region — batch interpretation**

Method: [impact-analysis.md#region-extent](../../../docs/design/reconstruction-impact/impact-analysis.md#region-extent)

### 4.3 Region extent by anatomy

**Method:** [Region extent](../../../docs/design/reconstruction-impact/impact-analysis.md#region-extent)

In [ ]:
REGION_ROWS = []
for parent, run in RUNS.items():
    spatial = run["spatial"]; metrics = spatial["metrics"].copy(); metrics["in_region"] = metrics["in_state_region"]
    extent = summarize_region_extent_by_anatomy(metrics, window_side_length=float(spatial["side_um"])).assign(scope=parent, scale_um=spatial["side_um"], threshold_status=spatial["state_threshold"]["status"])
    metrics = metrics.drop(columns="in_region")
    spatial.update(metrics=metrics, extent=extent)
    run["impact"].region_extent_by_anatomy = extent
    local_dir = ANALYSIS_ROOT / "local_state" / parent
    metrics.to_csv(local_dir / "window_metrics_with_state_region.csv", index=False)
    extent.to_csv(local_dir / "state_region_extent_by_anatomy.csv", index=False)
    gain_extent = summarize_region_extent_by_anatomy(metrics.assign(in_region=metrics.in_gain_region), window_side_length=float(spatial["side_um"]))
    gain_extent.to_csv(local_dir / "gain_region_extent_audit.csv", index=False)
    display(extent.loc[extent.level1_region.isin(["Overall", "Tumor", "Normal", "Interface"])])
    REGION_ROWS.append(extent.assign(parent=parent))
REGION_EXTENT = pd.concat(REGION_ROWS, ignore_index=True)

<!-- impact-node:4.3 -->
**Region coverage — batch interpretation**

Method: [impact-analysis.md#region-extent](../../../docs/design/reconstruction-impact/impact-analysis.md#region-extent)

#### Threshold audit

**Method:** [Region reliability](../../../docs/design/reconstruction-impact/impact-analysis.md#region-reliability)

In [ ]:
for parent, run in RUNS.items():
    spatial = run["spatial"]; local_dir = ANALYSIS_ROOT / "local_state" / parent
    spatial["state_bootstrap"].to_csv(local_dir / "state_threshold_bootstrap.csv", index=False)
    spatial["gain_bootstrap"].to_csv(local_dir / "gain_threshold_bootstrap_audit.csv", index=False)
    pd.DataFrame([spatial["state_threshold"]]).to_json(local_dir / "state_threshold.json", orient="records", indent=2)
    pd.DataFrame([spatial["gain_threshold"] | {"role": "audit_only"}]).to_json(local_dir / "gain_threshold_audit.json", orient="records", indent=2)
    display(pd.DataFrame([spatial["state_threshold"]])); display(spatial["state_bootstrap"].head())

### 4.4 Scale sensitivity

**Method:** [Scale sensitivity](../../../docs/design/reconstruction-impact/impact-analysis.md#scale-sensitivity)

In [ ]:
SCALE_ROWS = []
for parent, run in RUNS.items():
    spatial = run["spatial"]; rows = []
    for side_um in CONFIG["spatial_region"]["candidate_window_sides_um"]:
        windows = assign_square_windows(spatial["paired_um"], window_side_length=float(side_um), origin=spatial["origin"])
        values = compute_rarefied_window_diversity(windows, spatial["assignments"].raw_cluster, spatial["assignments"].recon_cluster, raw_level2_labels=spatial["level2"].labels, min_parent_units=4, n_draws=200, random_state=SEED)
        valid = values.loc[values.valid_window]
        row = {"parent": parent, "window_side_length": float(side_um), "n_valid_windows": len(valid), "min_parent_units": 4, "rarefaction_draws": 200, "random_state": SEED}
        for metric in ("k_obs", "neff", "evenness"):
            for suffix in ("raw", "recon", "level2"):
                row[f"median_{metric}_{suffix}"] = valid[f"{metric}_{suffix}"].median()
            for baseline in ("raw_leiden", "raw_level2"):
                row[f"median_delta_{metric}_vs_{baseline}"] = valid[f"delta_{metric}_vs_{baseline}"].median()
        rows.append(row)
    sensitivity = pd.DataFrame(rows); sensitivity.to_csv(ANALYSIS_ROOT / "local_state" / parent / "scale_sensitivity.csv", index=False)
    SCALE_ROWS.append(sensitivity); display(sensitivity)
SCALE_SENSITIVITY = pd.concat(SCALE_ROWS, ignore_index=True)
SCALE_SENSITIVITY.to_csv(ANALYSIS_ROOT / "local_state" / "scale_sensitivity_all_parents.csv", index=False)

<!-- impact-node:4.4 -->
**Scale sensitivity — batch interpretation**

Method: [impact-analysis.md#scale-sensitivity](../../../docs/design/reconstruction-impact/impact-analysis.md#scale-sensitivity)

### 4.5 Region conclusion

**Method:** [Region reliability](../../../docs/design/reconstruction-impact/impact-analysis.md#region-reliability)

<!-- impact-node:4.5 -->
**Region conclusion — batch interpretation**

Method: [outputs-and-test-plan.md](../../../docs/design/reconstruction-impact/outputs-and-test-plan.md)

## 5. Cross-parent summary and evidence boundary

**Method:** [Comparison foundation](../../../docs/design/reconstruction-impact/input-views.md#comparison-foundation); [Region reliability](../../../docs/design/reconstruction-impact/impact-analysis.md#region-reliability)

In [ ]:
SOURCE_FILES["content_contract.py"] = ROOT / "reproduce/case/reconstruction_impact/content_contract.py"
FINAL_EVIDENCE = save_final_evidence(
    RUNS, CONFIG, OUTPUT_DIR, COMPARISONS, change_table, MORAN_SUMMARY,
    aucell_summary, SAMPLE_ID, EMT_NAME, GMT_PATH, SOURCE_FILES, figures,
)
LOCAL_DIVERSITY_CROSS_PARENT = FINAL_EVIDENCE["local_diversity"]
REGION_EXTENT_CROSS_PARENT = FINAL_EVIDENCE["region_extent"]
LOCAL_WIDE = FINAL_EVIDENCE["local_wide"]
CROSS_PARENT = FINAL_EVIDENCE["cross_parent"]
ARTIFACTS = FINAL_EVIDENCE["artifacts"]
display(CROSS_PARENT.round(4))
display(LOCAL_DIVERSITY_CROSS_PARENT.round(4))
display(REGION_EXTENT_CROSS_PARENT.loc[
    REGION_EXTENT_CROSS_PARENT["level1_region"].isin(["Overall", "Tumor", "Normal", "Interface"])
].round(4))

In [ ]:
display(ARTIFACTS)
raw_context.file.close()
if "recon_context" in globals():
    recon_context.file.close()

<!-- impact-node:summary -->
**Cross-parent interpretation — batch interpretation**

Method: [outputs-and-test-plan.md](../../../docs/design/reconstruction-impact/outputs-and-test-plan.md)